Carregamento de tabelas do azura para o catalog

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/00_Utils

In [0]:
dbutils.fs.unmount("/mnt/working_data")


In [0]:
storage_account_name = "aimasterdata"
container_name = "working-data"
sas_token = "xxx"
mount_point = "/mnt/working_data"

dbutils.fs.mount(
    source = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net",
    mount_point = "/mnt/working_data",
    extra_configs = {f"fs.azure.sas.{container_name}.{storage_account_name}.blob.core.windows.net": sas_token}
)

# Verify mount
display(dbutils.fs.ls(mount_point))

In [0]:
dbutils.fs.cp("dbfs:/mnt/working_data/ARQLMED.zip", "file:/tmp/ARQLMED.zip")

In [0]:
# Unzip the file in the local file system
with zipfile.ZipFile("/tmp/ARQLMED.zip", "r") as zip_ref:
    zip_ref.extractall("/tmp/working_data/")

# with zipfile.ZipFile("/tmp/EVENT_LOG.zip", "r") as zip_ref:
#     zip_ref.extractall("/tmp/working_data/")

# Move the extracted files to DBFS
dbutils.fs.cp("file:/tmp/working_data/archives/export/ARQLMED.csv", "dbfs:/mnt/working_data/working_arqlmed.csv")
#dbutils.fs.cp("file:/tmp/working_data/MEDIDAS.csv", "dbfs:/mnt/working_data/working_medidas")
#dbutils.fs.cp("file:/tmp/working_data/EVENT_LOG/EVENT_LOG.csv", "dbfs:/mnt/working_data/working_event_log")


In [0]:
# Read the Parquet files into DataFrames
df_arqlmed = spark.read.option("header", "true").option("delimiter", ";").csv("dbfs:/mnt/working_data/working_arqlmed.csv")

# Show a few rows to ensure the DataFrame is populated correctly
df_arqlmed.display(5)
df_arqlmed.printSchema()


In [0]:
# Read the Parquet files into DataFrames
df_arqlmed = spark.read.parquet("/mnt/working_data/working_arqlmed.parquet")


In [0]:

# File location and type
file_location = "dbfs:/mnt/working_data/working_arqlmed.csv"
file_type = "csv"

# CSV options
infer_schema = "false"
first_row_is_header = "true"
delimiter = ";"

# The applied options are for CSV files. For other file types, these will be ignored.
df = spark.read.format(file_type) \
  .option("inferSchema", infer_schema) \
  .option("header", first_row_is_header) \
  .option("sep", delimiter) \
  .load(file_location)

display(df)

In [0]:
# Create a view or table

temp_table_name = "working_arqlmed"

df.createOrReplaceTempView(temp_table_name)

In [0]:
%sql

/* Query the created temp table in a SQL cell */

select * from `working_arqlmed`

In [0]:
# With this registered as a temp view, it will only be available to this particular notebook. If you'd like other users to be able to query this table, you can also create a table from the DataFrame.
# Once saved, this table will persist across cluster restarts as well as allow various users across different notebooks to query this data.
# To do so, choose your table name and uncomment the bottom line.

permanent_table_name = "working_arqlmed"

df.write.format("parquet").saveAsTable(permanent_table_name)

In [0]:

# File location and type
file_location = "dbfs:/mnt/working_data/working_medidas"
file_type = "csv"

# CSV options
infer_schema = "false"
first_row_is_header = "true"
delimiter = ";"

# The applied options are for CSV files. For other file types, these will be ignored.
df = spark.read.format(file_type) \
  .option("inferSchema", infer_schema) \
  .option("header", first_row_is_header) \
  .option("sep", delimiter) \
  .load(file_location)

display(df)

In [0]:
# Create a view or table

temp_table_name = "working_medidas"

df.createOrReplaceTempView(temp_table_name)

In [0]:
# With this registered as a temp view, it will only be available to this particular notebook. If you'd like other users to be able to query this table, you can also create a table from the DataFrame.
# Once saved, this table will persist across cluster restarts as well as allow various users across different notebooks to query this data.
# To do so, choose your table name and uncomment the bottom line.

permanent_table_name = "working_medidas"

df.write.format("parquet").saveAsTable(permanent_table_name)

In [0]:
# File location and type
file_location = "/FileStore/tables/medidas.csv"
file_type = "csv"

# CSV options
infer_schema = "false"
first_row_is_header = "true"
delimiter = ","

# The applied options are for CSV files. For other file types, these will be ignored.
df = spark.read.format(file_type) \
  .option("inferSchema", infer_schema) \
  .option("header", first_row_is_header) \
  .option("sep", delimiter) \
  .load(file_location)

display(df)